# Tutorial: LM-02 Logistic Regression

Audience:
- Learners studying canonical linear classification models.

Prerequisites:
- Linear regression as a baseline linear model.
- Bernoulli and categorical probabilities.
- Cross-entropy, KL divergence, and gradient descent.

Learning goals:
- Fit binary logistic regression and multiclass softmax regression from scratch.
- Inspect cross-entropy loss during optimization.
- Visualize linear decision boundaries on two-dimensional data.
- Connect probabilistic outputs to geometric classification regions.


## Outline

1. Implement binary logistic regression with gradient descent.
2. Fit it on a 2D binary dataset and visualize the decision boundary.
3. Implement multiclass softmax regression.
4. Fit it on a 2D three-class dataset and visualize the decision regions.
5. Compare the scratch implementations to `scikit-learn` as a quick sanity check.


In [1]:
from __future__ import annotations

import matplotlib

import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import make_blobs
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss

SEED = 17
rng = np.random.default_rng(SEED)
SEED

17

In [2]:
def add_bias(X: np.ndarray) -> np.ndarray:
    return np.column_stack([np.ones(X.shape[0]), X])


def sigmoid(z: np.ndarray) -> np.ndarray:
    z = np.clip(z, -40.0, 40.0)
    return 1.0 / (1.0 + np.exp(-z))


def binary_cross_entropy(y: np.ndarray, p: np.ndarray) -> float:
    p = np.clip(p, 1e-9, 1.0 - 1e-9)
    return float(-np.mean(y * np.log(p) + (1.0 - y) * np.log(1.0 - p)))


def fit_binary_logistic(
    X: np.ndarray,
    y: np.ndarray,
    learning_rate: float = 0.2,
    steps: int = 4000,
) -> tuple[np.ndarray, list[float]]:
    Xb = add_bias(X)
    theta = np.zeros(Xb.shape[1])
    losses: list[float] = []

    for step in range(steps):
        p = sigmoid(Xb @ theta)
        gradient = Xb.T @ (p - y) / Xb.shape[0]
        theta -= learning_rate * gradient
        if step % 40 == 0 or step == steps - 1:
            losses.append(binary_cross_entropy(y, sigmoid(Xb @ theta)))

    return theta, losses


def predict_binary_proba(X: np.ndarray, theta: np.ndarray) -> np.ndarray:
    return sigmoid(add_bias(X) @ theta)


def one_hot(y: np.ndarray, num_classes: int) -> np.ndarray:
    Y = np.zeros((y.size, num_classes))
    Y[np.arange(y.size), y] = 1.0
    return Y


def softmax(scores: np.ndarray) -> np.ndarray:
    shifted = scores - scores.max(axis=1, keepdims=True)
    exp_scores = np.exp(shifted)
    return exp_scores / exp_scores.sum(axis=1, keepdims=True)


def multiclass_cross_entropy(Y: np.ndarray, P: np.ndarray) -> float:
    P = np.clip(P, 1e-9, 1.0)
    return float(-np.mean(np.sum(Y * np.log(P), axis=1)))


def fit_softmax_regression(
    X: np.ndarray,
    y: np.ndarray,
    num_classes: int,
    learning_rate: float = 0.15,
    steps: int = 5000,
) -> tuple[np.ndarray, list[float]]:
    Xb = add_bias(X)
    Y = one_hot(y, num_classes)
    W = np.zeros((num_classes, Xb.shape[1]))
    losses: list[float] = []

    for step in range(steps):
        P = softmax(Xb @ W.T)
        gradient = ((P - Y).T @ Xb) / Xb.shape[0]
        W -= learning_rate * gradient
        if step % 50 == 0 or step == steps - 1:
            losses.append(multiclass_cross_entropy(Y, softmax(Xb @ W.T)))

    return W, losses


def predict_softmax_proba(X: np.ndarray, W: np.ndarray) -> np.ndarray:
    return softmax(add_bias(X) @ W.T)


def make_grid(X: np.ndarray, padding: float = 1.0, points: int = 250) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    x_min, x_max = X[:, 0].min() - padding, X[:, 0].max() + padding
    y_min, y_max = X[:, 1].min() - padding, X[:, 1].max() + padding
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, points),
        np.linspace(y_min, y_max, points),
    )
    grid = np.column_stack([xx.ravel(), yy.ravel()])
    return xx, yy, grid

## Binary logistic regression on 2D data

We build a mildly overlapping two-class problem. Because the features are two-dimensional, the learned probability surface and the `0.5` decision boundary can be plotted directly.


In [3]:
X_bin, y_bin = make_blobs(
    n_samples=280,
    centers=[(-2.2, -1.5), (2.0, 1.8)],
    cluster_std=[1.4, 1.6],
    random_state=SEED,
)
y_bin = y_bin.astype(float)

theta_bin, losses_bin = fit_binary_logistic(X_bin, y_bin)
p_bin = predict_binary_proba(X_bin, theta_bin)
y_hat_bin = (p_bin >= 0.5).astype(float)

print('scratch theta:', np.round(theta_bin, 3))
print('scratch training accuracy:', round(float(accuracy_score(y_bin, y_hat_bin)), 4))
print('scratch training log loss:', round(binary_cross_entropy(y_bin, p_bin), 4))

scratch theta: [-0.044  1.681  1.843]
scratch training accuracy: 0.9571
scratch training log loss: 0.098


In [4]:
model_bin = LogisticRegression(random_state=SEED)
model_bin.fit(X_bin, y_bin.astype(int))
p_bin_sklearn = model_bin.predict_proba(X_bin)[:, 1]

print('sklearn training accuracy:', round(float(model_bin.score(X_bin, y_bin)), 4))
print('sklearn training log loss:', round(float(log_loss(y_bin, p_bin_sklearn)), 4))

sklearn training accuracy: 0.9607
sklearn training log loss: 0.0991


In [5]:
xx, yy, grid = make_grid(X_bin)
grid_prob = predict_binary_proba(grid, theta_bin).reshape(xx.shape)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

scatter = axes[0].scatter(
    X_bin[:, 0],
    X_bin[:, 1],
    c=y_bin,
    cmap='coolwarm',
    edgecolor='black',
    alpha=0.85,
)
axes[0].contour(xx, yy, grid_prob, levels=[0.5], colors='black', linewidths=2)
axes[0].contourf(xx, yy, grid_prob, levels=np.linspace(0, 1, 12), cmap='coolwarm', alpha=0.25)
axes[0].set_title('Binary logistic regression decision boundary')
axes[0].set_xlabel('x1')
axes[0].set_ylabel('x2')

axes[1].plot(np.arange(len(losses_bin)) * 40, losses_bin, color='darkgreen', linewidth=2)
axes[1].set_title('Binary cross-entropy during training')
axes[1].set_xlabel('gradient step')
axes[1].set_ylabel('loss')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

C:\Users\noaha\AppData\Local\Temp\ipykernel_22400\2868363143.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


The plot shows the usual pattern: probabilities vary smoothly across the plane, but the class boundary itself is linear because it is defined by a threshold on a linear score. Loss decreases quickly and then flattens as gradient descent approaches a good solution.


## Multiclass softmax regression on 2D data

We now switch to three classes. Each class gets its own linear score, and softmax turns those scores into a probability vector that sums to one.


In [6]:
X_multi, y_multi = make_blobs(
    n_samples=360,
    centers=[(-3.0, 2.2), (0.0, -2.4), (3.1, 2.6)],
    cluster_std=[1.2, 1.3, 1.1],
    random_state=SEED + 1,
)

W_multi, losses_multi = fit_softmax_regression(X_multi, y_multi, num_classes=3)
P_multi = predict_softmax_proba(X_multi, W_multi)
y_hat_multi = P_multi.argmax(axis=1)

print('scratch training accuracy:', round(float(accuracy_score(y_multi, y_hat_multi)), 4))
print('scratch training cross-entropy:', round(multiclass_cross_entropy(one_hot(y_multi, 3), P_multi), 4))
print('first five predicted probability vectors:')
print(np.round(P_multi[:5], 3))

scratch training accuracy: 0.9806
scratch training cross-entropy: 0.0347
first five predicted probability vectors:
[[0.929 0.    0.071]
 [0.    0.    1.   ]
 [0.    1.    0.   ]
 [0.    1.    0.   ]
 [1.    0.    0.   ]]


In [7]:
model_multi = LogisticRegression(
    solver='lbfgs',
    random_state=SEED,
    max_iter=2000,
)
model_multi.fit(X_multi, y_multi)
P_multi_sklearn = model_multi.predict_proba(X_multi)

print('sklearn training accuracy:', round(float(model_multi.score(X_multi, y_multi)), 4))
print('sklearn training cross-entropy:', round(float(log_loss(y_multi, P_multi_sklearn)), 4))

sklearn training accuracy: 0.9806
sklearn training cross-entropy: 0.0408


In [8]:
xx_m, yy_m, grid_m = make_grid(X_multi)
grid_pred = predict_softmax_proba(grid_m, W_multi).argmax(axis=1).reshape(xx_m.shape)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].contourf(xx_m, yy_m, grid_pred, levels=np.arange(-0.5, 3.5, 1), cmap='Set2', alpha=0.35)
axes[0].scatter(
    X_multi[:, 0],
    X_multi[:, 1],
    c=y_multi,
    cmap='Set2',
    edgecolor='black',
    alpha=0.9,
)
axes[0].set_title('Softmax regression decision regions')
axes[0].set_xlabel('x1')
axes[0].set_ylabel('x2')

axes[1].plot(np.arange(len(losses_multi)) * 50, losses_multi, color='purple', linewidth=2)
axes[1].set_title('Multiclass cross-entropy during training')
axes[1].set_xlabel('gradient step')
axes[1].set_ylabel('loss')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

C:\Users\noaha\AppData\Local\Temp\ipykernel_22400\182327804.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Each region is still separated by linear score comparisons, so the boundaries are piecewise linear. Near a boundary, class probabilities are less extreme because competing class scores are close; deep inside a region, the winning class probability becomes dominant.


## Takeaways

- Binary logistic regression fits a Bernoulli model with probabilities produced by a sigmoid.
- Multiclass logistic regression fits a categorical model with probabilities produced by softmax.
- Cross-entropy is the natural training objective because it is negative log-likelihood.
- The probability map is nonlinear, but the induced score-based boundaries are linear in the input features.
